In [ ]:
%pip install pandas openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.7 MB 10.1 MB/s eta 0:00:01
   ----------------- ---------------------- 4.2/9.7 MB 11.0 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.7 MB 10.3 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.7 MB 9.9 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.7 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 9.1 MB/s eta 0:00:00
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 6.7 MB/s eta 0:00:02
   --------- ------------------------------ 2.9/12.6 MB 7.6 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.6 M


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Understanding data and finding anomalies

In [ ]:
import pandas as pd

# Read CSV
df = pd.read_csv("claims.csv")

# Print column name and data type
print(f"Total columns: {len(df.columns)}\n")

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col} --> {df[col].dtype}")

Total columns: 31

1. claim_ref --> str
2. source_system --> str
3. dateOfLoss --> str
4. yearOfLoss --> int64
5. state --> str
6. reportedZipCode --> str
7. countyCode --> float64
8. latitude --> float64
9. longitude --> float64
10. floodEvent --> str
11. causeOfDamage --> str
12. ratedFloodZone --> str
13. occupancyType --> int64
14. numberOfFloorsInTheInsuredBuilding --> float64
15. originalConstructionDate --> str
16. elevatedBuildingIndicator --> int64
17. primaryResidenceIndicator --> int64
18. totalBuildingInsuranceCoverage --> float64
19. totalContentsInsuranceCoverage --> float64
20. buildingPropertyValue --> float64
21. buildingReplacementCost --> float64
22. buildingDamageAmount --> float64
23. contentsDamageAmount --> float64
24. waterDepth --> float64
25. floodWaterDuration --> float64
26. amountPaidOnBuildingClaim --> float64
27. amountPaidOnContentsClaim --> float64
28. netBuildingPaymentAmount --> float64
29. nonPaymentReasonBuilding --> float64
30. openDate --> str
31.

In [ ]:
df.head()

,claim_ref,source_system,dateOfLoss,yearOfLoss,state,reportedZipCode,countyCode,latitude,longitude,floodEvent,...,buildingDamageAmount,contentsDamageAmount,waterDepth,floodWaterDuration,amountPaidOnBuildingClaim,amountPaidOnContentsClaim,netBuildingPaymentAmount,nonPaymentReasonBuilding,openDate,mostRecentPaymentDate
0,CLM-6119947,core,2018-04-13T00:00:00.000Z,2018,HI,96734,15003.0,21.4,-157.7,NaN,...,15092.0,NaN,NaN,0.0,13841.58,0.0,13841.58,NaN,2018-04-26T00:00:00.000Z,2018-05-21T00:00:00.000Z
1,CLM-5642877,core,2021-09-01T00:00:00.000Z,2021,MA,1906,25009.0,42.4,-71.0,Hurricane Ida,...,0.0,0.0,NaN,NaN,0.00,0.0,0.00,6.0,2021-09-02T00:00:00.000Z,2021-09-22T00:00:00.000Z
2,CLM-7518535,core,2020-05-20T00:00:00.000Z,2020,MI,48640,26111.0,43.6,-84.3,NaN,...,44478.0,NaN,2.0,NaN,5000.00,0.0,5000.00,NaN,2020-05-20T00:00:00.000Z,2020-06-18T00:00:00.000Z
3,CLM-6796433,core,2024-09-26T00:00:00.000Z,2024,FL,33634,12057.0,28.0,-82.6,Hurricane Helene,...,100212.0,NaN,1.0,NaN,90211.66,0.0,90211.66,NaN,2024-09-30T00:00:00.000Z,2024-12-04T00:00:00.000Z
4,CLM-5558178,core,2024-09-26T00:00:00.000Z,2024,FL,33710,12103.0,27.8,-82.8,Hurricane Helene,...,153834.0,NaN,2.0,NaN,143833.51,0.0,143833.51,NaN,2024-09-28T00:00:00.000Z,2025-02-23T00:00:00.000Z


In [ ]:
print("Dataset size:", df.shape)

Dataset size: (2040, 31)


In [ ]:
# Missing values per column
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": df.isna().mean() * 100
})

# Sort by highest missing percentage
missing = missing.sort_values("Missing %", ascending=False)

print(missing)

                                    Missing Count  Missing %
nonPaymentReasonBuilding                     1651  80.931373
floodWaterDuration                           1609  78.872549
contentsDamageAmount                          904  44.313725
floodEvent                                    564  27.647059
waterDepth                                    308  15.098039
buildingDamageAmount                          220  10.784314
buildingPropertyValue                         220  10.784314
buildingReplacementCost                       220  10.784314
amountPaidOnContentsClaim                     194   9.509804
mostRecentPaymentDate                         194   9.509804
amountPaidOnBuildingClaim                     192   9.411765
causeOfDamage                                  29   1.421569
originalConstructionDate                       14   0.686275
latitude                                       14   0.686275
longitude                                      14   0.686275
numberOfFloorsInTheInsur

In [ ]:
# Separate columns by type
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

print("Categorical:")
print(categorical_cols)

print("\nNumeric:")
print(numeric_cols)

# Unique values for categorical columns
for col in categorical_cols:
    print(f"\n{'='*60}")
    print(col)
    print("Unique:", df[col].nunique())
    print(df[col].value_counts(dropna=False).head(30))

Categorical:
['claim_ref', 'source_system', 'dateOfLoss', 'state', 'reportedZipCode', 'floodEvent', 'causeOfDamage', 'ratedFloodZone', 'originalConstructionDate', 'openDate', 'mostRecentPaymentDate']

Numeric:
['yearOfLoss', 'countyCode', 'latitude', 'longitude', 'occupancyType', 'numberOfFloorsInTheInsuredBuilding', 'elevatedBuildingIndicator', 'primaryResidenceIndicator', 'totalBuildingInsuranceCoverage', 'totalContentsInsuranceCoverage', 'buildingPropertyValue', 'buildingReplacementCost', 'buildingDamageAmount', 'contentsDamageAmount', 'waterDepth', 'floodWaterDuration', 'amountPaidOnBuildingClaim', 'amountPaidOnContentsClaim', 'netBuildingPaymentAmount', 'nonPaymentReasonBuilding']

claim_ref
Unique: 2040
claim_ref
CLM-6119947      1
CLM-5642877      1
CLM-7518535      1
CLM-6796433      1
CLM-5558178      1
CLM-7658990      1
CLM-6515202      1
CLM-7046214      1
CLM-5976757      1
CLM-5813447      1
CLM-6726350      1
CLM-6830121      1
CLM-7820778      1
CLM-7822961      1
CLM-5

C:\Users\SIDDESHWAR\AppData\Local\Temp\ipykernel_35820\975322498.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
date_cols = [
    "dateOfLoss",
    "originalConstructionDate",
    "openDate",
    "mostRecentPaymentDate"
]

for col in date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce", utc=True)

    print(f"\n{'='*70}")
    print(col)
    print("Original non-null:", df[col].notna().sum())
    print("Failed to parse:", parsed.notna().sum(), "success")
    print("Min:", parsed.min())
    print("Max:", parsed.max())


dateOfLoss
Original non-null: 2040
Failed to parse: 1917 success
Min: 2018-01-04 00:00:00+00:00
Max: 2026-08-20 00:00:00+00:00

originalConstructionDate
Original non-null: 2026
Failed to parse: 2026 success
Min: 1492-10-12 00:00:00+00:00
Max: 2025-06-01 00:00:00+00:00

openDate
Original non-null: 2040
Failed to parse: 2040 success
Min: 2018-01-05 00:00:00+00:00
Max: 2026-09-03 00:00:00+00:00

mostRecentPaymentDate
Original non-null: 1846
Failed to parse: 1846 success
Min: 1942-10-12 00:00:00+00:00
Max: 2026-09-06 00:00:00+00:00


In [ ]:
for col in date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce", utc=True)

    print(f"\n{col} suspicious / failed:")
    print(df.loc[parsed.isna() & df[col].notna(), ["claim_ref", col]].head(30))


dateOfLoss suspicious / failed:
         claim_ref  dateOfLoss
40     CLM-7898726  29-09-2023
67     CLM-5746454  30-08-2023
82     CLM-6157209  10-10-2024
99     CLM-6470168  09-10-2024
144    CLM-6398916  30-09-2022
154    CLM-7011673  05-04-2025
156    CLM-5725175  23-03-2026
158    CLM-5848380  30-09-2022
165    CLM-8169782  19-05-2020
168    CLM-6916300  21-04-2025
186  CLM-7127665-R  03-09-2018
211    CLM-6799102  04-08-2019
235    CLM-6409025  21-08-2025
250    CLM-7337861  26-09-2024
318    CLM-6134163  26-09-2024
330    CLM-7980100  05-09-2019
345    CLM-5897469  14-09-2018
364    CLM-8106199  13-06-2024
379    CLM-7326132  13-09-2018
385    CLM-7159386  11-01-2020
393    CLM-7701299  06-09-2019
432    CLM-6877029  21-04-2025
456    CLM-6304365  27-03-2026
471    CLM-7648582  28-09-2022
479    CLM-8024729  16-02-2021
484    CLM-6410405  20-03-2019
489    CLM-6729784  23-09-2018
504    CLM-6134144  05-09-2020
521    CLM-5597390  20-08-2023
527    CLM-6611416  27-03-2021

origi

In [ ]:
df_check = df.copy()

for col in [
    "dateOfLoss",
    "originalConstructionDate",
    "openDate",
    "mostRecentPaymentDate"
]:
    df_check[col + "_parsed"] = pd.to_datetime(
        df_check[col],
        errors="coerce",
        dayfirst=True,
        utc=True
    )

C:\Users\SIDDESHWAR\AppData\Local\Temp\ipykernel_35820\1963631726.py:9: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_check[col + "_parsed"] = pd.to_datetime(
C:\Users\SIDDESHWAR\AppData\Local\Temp\ipykernel_35820\1963631726.py:9: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_check[col + "_parsed"] = pd.to_datetime(
C:\Users\SIDDESHWAR\AppData\Local\Temp\ipykernel_35820\1963631726.py:9: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_check[col + "_parsed"] = pd.to_datetime(


In [ ]:
bad_open = (
    df_check["openDate_parsed"].notna() &
    df_check["dateOfLoss_parsed"].notna() &
    (df_check["openDate_parsed"] < df_check["dateOfLoss_parsed"])
)

print("Open date before loss:", bad_open.sum())

print(
    df_check.loc[
        bad_open,
        ["claim_ref", "dateOfLoss", "openDate"]
    ].head(30)
)

Open date before loss: 0
Empty DataFrame
Columns: [claim_ref, dateOfLoss, openDate]
Index: []


In [ ]:
bad_payment = (
    df_check["mostRecentPaymentDate_parsed"].notna() &
    (df_check["dateOfLoss_parsed"].notna()) &
    (
        df_check["mostRecentPaymentDate_parsed"]
        < df_check["dateOfLoss_parsed"]
    )
)

print("Payment before loss:", bad_payment.sum())

print(
    df_check.loc[
        bad_payment,
        ["claim_ref", "dateOfLoss", "mostRecentPaymentDate"]
    ].head(30)
)

Payment before loss: 6
        claim_ref                dateOfLoss     mostRecentPaymentDate
8     CLM-5976757  2020-07-26T00:00:00.000Z  1942-10-12T00:00:00.000Z
134   CLM-6830799  2019-09-17T00:00:00.000Z  1942-10-12T00:00:00.000Z
199   CLM-5629942  2020-07-01T00:00:00.000Z  1942-10-12T00:00:00.000Z
501   CLM-5611558  2019-10-27T00:00:00.000Z  1942-10-12T00:00:00.000Z
723   CLM-7023722  2020-07-23T00:00:00.000Z  1942-10-12T00:00:00.000Z
1035  CLM-7769047  2020-03-24T00:00:00.000Z  1942-10-12T00:00:00.000Z


In [ ]:
bad_construction = (
    df_check["originalConstructionDate_parsed"].notna() &
    df_check["dateOfLoss_parsed"].notna() &
    (
        df_check["originalConstructionDate_parsed"]
        > df_check["dateOfLoss_parsed"]
    )
)

print("Construction after loss:", bad_construction.sum())

print(
    df_check.loc[
        bad_construction,
        ["claim_ref", "dateOfLoss", "originalConstructionDate"]
    ].head(30)
)

Construction after loss: 10
        claim_ref                dateOfLoss  originalConstructionDate
142   CLM-6338089  2024-09-20T00:00:00.000Z  2025-06-01T00:00:00.000Z
310   CLM-6153716  2022-09-28T00:00:00.000Z  2025-06-01T00:00:00.000Z
825   CLM-6454802  2020-11-09T00:00:00.000Z  2025-06-01T00:00:00.000Z
1020  CLM-6108302  2022-09-28T00:00:00.000Z  2025-06-01T00:00:00.000Z
1021  CLM-7892430  2022-09-28T00:00:00.000Z  2025-06-01T00:00:00.000Z
1186  CLM-6910176  2018-10-11T00:00:00.000Z  2025-06-01T00:00:00.000Z
1474  CLM-7453656  2021-06-18T00:00:00.000Z  2025-06-01T00:00:00.000Z
1480  CLM-6767884  2020-05-25T00:00:00.000Z  2025-06-01T00:00:00.000Z
1555  CLM-7282073  2022-09-28T00:00:00.000Z  2025-06-01T00:00:00.000Z
1878  CLM-7952766  2022-01-01T00:00:00.000Z  2025-06-01T00:00:00.000Z


# Issues found so far
1. Some rated flood zone are in smaller case and others in uppercase
2. Date format inconsistency, espesially in dateOfLoss
3. payment date < loss date - 6 rows (1942-10-12)
4. construction date > loss date - 10 rowa (2025-06-01)


In [13]:
print(
    sorted(
        df["ratedFloodZone"]
        .dropna()
        .astype(str)
        .unique()
    )
)

['A', 'A01', 'A02', 'A03', 'A04', 'A05', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A15', 'A18', 'A99', 'AE', 'AH', 'AHB', 'AO', 'AOB', 'B', 'C', 'D', 'V16', 'VE', 'X', 'a ', 'ae ', 'ah ', 'ao ', 'x ']


In [14]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

print(
    df[
        ~df["ratedFloodZone"]
        .fillna("")
        .astype(str)
        .str.upper()
        .isin([
            # don't fill this list yet
        ])
    ][["claim_ref", "ratedFloodZone"]].to_string(index=False)
)

    claim_ref ratedFloodZone
  CLM-6119947              X
  CLM-5642877             AE
  CLM-7518535             AE
  CLM-6796433             AE
  CLM-5558178             VE
  CLM-7658990              A
  CLM-6515202             AE
  CLM-7046214             AE
  CLM-5976757              X
  CLM-5813447              X
  CLM-6726350             AE
  CLM-6830121              X
  CLM-7820778             AE
  CLM-7822961              X
  CLM-5576313             AE
  CLM-7455343             AE
  CLM-8058795              X
  CLM-6596012             AE
  CLM-7062825             AE
  CLM-6053933             AE
  CLM-6223381             AE
  CLM-5637999              X
  CLM-7061589             AE
  CLM-7806166              X
  CLM-6278537              X
  CLM-8168400              A
  CLM-7130407              X
  CLM-6166678             AE
  CLM-8054536             VE
CLM-6839569-R             AE
  CLM-6701611            ae 
  CLM-7934685              X
  CLM-6127266             AE
  CLM-6678927 

# Clean 1
Converted all lower case to upper case in ratedFloodZone

In [15]:
df["ratedFloodZone"] = (
    df["ratedFloodZone"]
    .str.strip()
    .str.upper()
)

In [16]:
suspicious_zones = ["AHB", "AOB"]

print(
    df[df["ratedFloodZone"].isin(suspicious_zones)][
        ["claim_ref", "source_system", "state",
         "reportedZipCode", "ratedFloodZone"]
    ]
)

       claim_ref source_system state reportedZipCode ratedFloodZone
66   CLM-7986783          core    TX           78503            AHB
879  CLM-7990756          core    CA           93065            AOB


In [17]:
print(
    df["ratedFloodZone"]
    .value_counts(dropna=False)
    .sort_index()
)

ratedFloodZone
A        96
A01       2
A02       1
A03       1
A04       3
A05       2
A07       2
A08       6
A09       3
A10       4
A11       1
A12       2
A13       4
A15       2
A18       1
A99       5
AE     1295
AH       33
AHB       1
AO       21
AOB       1
B        12
C        26
D         1
V16       1
VE       66
X       443
NaN       5
Name: count, dtype: int64


In [18]:
parsed_loss = pd.to_datetime(
    df["dateOfLoss"],
    errors="coerce",
    utc=True
)

failed = parsed_loss.isna()

print(
    df.loc[
        failed,
        ["claim_ref", "source_system", "dateOfLoss", "yearOfLoss"]
    ].head(20)
)

print("\nSource distribution:")
print(df.loc[failed, "source_system"].value_counts())

         claim_ref source_system  dateOfLoss  yearOfLoss
40     CLM-7898726    legacy_bdx  29-09-2023        2023
67     CLM-5746454    legacy_bdx  30-08-2023        2023
82     CLM-6157209    legacy_bdx  10-10-2024        2024
99     CLM-6470168    legacy_bdx  09-10-2024        2024
144    CLM-6398916    legacy_bdx  30-09-2022        2022
154    CLM-7011673    legacy_bdx  05-04-2025        2025
156    CLM-5725175    legacy_bdx  23-03-2026        2026
158    CLM-5848380    legacy_bdx  30-09-2022        2022
165    CLM-8169782    legacy_bdx  19-05-2020        2020
168    CLM-6916300    legacy_bdx  21-04-2025        2025
186  CLM-7127665-R    legacy_bdx  03-09-2018        2018
211    CLM-6799102    legacy_bdx  04-08-2019        2019
235    CLM-6409025    legacy_bdx  21-08-2025        2025
250    CLM-7337861    legacy_bdx  26-09-2024        2024
318    CLM-6134163    legacy_bdx  26-09-2024        2024
330    CLM-7980100    legacy_bdx  05-09-2019        2019
345    CLM-5897469    legacy_bd

# Clean 2

Because the exact same old date appears repeatedly and creates the violation, this is strong evidence that 1942-10-12 is a placeholder/default rather than a genuine payment date.

So I decided to change them because they it is pracitcally not possible so i set them to Nan.

In [19]:
result = df[
    pd.to_datetime(df["mostRecentPaymentDate"], errors="coerce").dt.date
    == pd.Timestamp("1942-10-12").date()
]

print(
    result[
        [
            "claim_ref",
            "dateOfLoss",
            "mostRecentPaymentDate",
            "amountPaidOnBuildingClaim",
            "amountPaidOnContentsClaim",
            "netBuildingPaymentAmount",
            "nonPaymentReasonBuilding",
            "source_system"
        ]
    ].to_string(index=False)
)

  claim_ref               dateOfLoss    mostRecentPaymentDate  amountPaidOnBuildingClaim  amountPaidOnContentsClaim  netBuildingPaymentAmount  nonPaymentReasonBuilding source_system
CLM-5976757 2020-07-26T00:00:00.000Z 1942-10-12T00:00:00.000Z                        0.0                        0.0                       0.0                       1.0          core
CLM-6830799 2019-09-17T00:00:00.000Z 1942-10-12T00:00:00.000Z                        0.0                        0.0                       0.0                       1.0          core
CLM-5629942 2020-07-01T00:00:00.000Z 1942-10-12T00:00:00.000Z                        0.0                        0.0                       0.0                       1.0          core
CLM-5611558 2019-10-27T00:00:00.000Z 1942-10-12T00:00:00.000Z                        0.0                        0.0                       0.0                      99.0          core
CLM-7023722 2020-07-23T00:00:00.000Z 1942-10-12T00:00:00.000Z                        0.0  

In [20]:
df_clean = df.copy()

In [21]:
import pandas as pd

# Work with the original strings
raw_loss = df_clean["dateOfLoss"].astype("string").str.strip()

# Create an empty datetime Series with UTC timezone
parsed_loss = pd.Series(
    pd.NaT,
    index=df_clean.index,
    dtype="datetime64[ns, UTC]"
)

# Identify the two formats
iso_mask = raw_loss.str.match(r"^\d{4}-\d{2}-\d{2}T")
legacy_mask = raw_loss.str.match(r"^\d{2}/\d{2}/\d{4}$")

# Parse ISO dates
parsed_loss.loc[iso_mask] = pd.to_datetime(
    raw_loss.loc[iso_mask],
    format="%Y-%m-%dT%H:%M:%S.%fZ",
    errors="coerce",
    utc=True
)

# Parse legacy DD/MM/YYYY dates
parsed_loss.loc[legacy_mask] = pd.to_datetime(
    raw_loss.loc[legacy_mask],
    format="%d/%m/%Y",
    errors="coerce",
    utc=True
)

# Replace the original column
df_clean["dateOfLoss"] = parsed_loss

In [22]:
for col in [
    "originalConstructionDate",
    "openDate",
    "mostRecentPaymentDate"
]:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce",
        utc=True
    )

In [23]:
date_cols = [
    "dateOfLoss",
    "originalConstructionDate",
    "openDate",
    "mostRecentPaymentDate"
]

for col in date_cols:
    print(
        col,
        "dtype =", df_clean[col].dtype,
        "| missing =", df_clean[col].isna().sum()
    )

dateOfLoss dtype = datetime64[ns, UTC] | missing = 123
originalConstructionDate dtype = datetime64[us, UTC] | missing = 14
openDate dtype = datetime64[us, UTC] | missing = 0
mostRecentPaymentDate dtype = datetime64[us, UTC] | missing = 194


In [24]:
mask = df_clean["mostRecentPaymentDate"] == pd.Timestamp(
    "1942-10-12",
    tz="UTC"
)

df_clean.loc[mask, "mostRecentPaymentDate"] = pd.NaT

In [25]:
print(
    "Remaining 1942-10-12:",
    (df_clean["mostRecentPaymentDate"] ==
     pd.Timestamp("1942-10-12", tz="UTC")).sum()
)

Remaining 1942-10-12: 0


In [26]:
mask = df_clean["originalConstructionDate"] == pd.Timestamp(
    "2025-06-01",
    tz="UTC"
)

df_clean.loc[mask, "originalConstructionDate"] = pd.NaT

In [27]:
print(
    "Payment before loss:",
    (df_clean["mostRecentPaymentDate"] <
     df_clean["dateOfLoss"]).sum()
)

print(
    "Construction after loss:",
    (df_clean["originalConstructionDate"] >
     df_clean["dateOfLoss"]).sum()
)

Payment before loss: 0
Construction after loss: 0


In [28]:
numeric_cols = [
    "countyCode",
    "latitude",
    "longitude",
    "occupancyType",
    "numberOfFloorsInTheInsuredBuilding",
    "elevatedBuildingIndicator",
    "primaryResidenceIndicator",
    "totalBuildingInsuranceCoverage",
    "totalContentsInsuranceCoverage",
    "buildingPropertyValue",
    "buildingReplacementCost",
    "buildingDamageAmount",
    "contentsDamageAmount",
    "waterDepth",
    "floodWaterDuration",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount",
    "nonPaymentReasonBuilding"
]

for col in numeric_cols:
    print(f"\n--- {col} ---")
    print("min:", df_clean[col].min())
    print("max:", df_clean[col].max())
    print("negative:", (df_clean[col] < 0).sum())
    print("zero:", (df_clean[col] == 0).sum())


--- countyCode ---
min: 1003.0
max: 72061.0
negative: 0
zero: 0

--- latitude ---
min: -93.5
max: 49.0
negative: 8
zero: 0

--- longitude ---
min: -159.5
max: 41.6
negative: 2018
zero: 0

--- occupancyType ---
min: 1
max: 18
negative: 0
zero: 0

--- numberOfFloorsInTheInsuredBuilding ---
min: 1.0
max: 5.0
negative: 0
zero: 0

--- elevatedBuildingIndicator ---
min: 0
max: 1
negative: 0
zero: 1579

--- primaryResidenceIndicator ---
min: 0
max: 1
negative: 0
zero: 659

--- totalBuildingInsuranceCoverage ---
min: 14.0
max: 20165000.0
negative: 0
zero: 0

--- totalContentsInsuranceCoverage ---
min: 0.0
max: 500000.0
negative: 0
zero: 627

--- buildingPropertyValue ---
min: 0.0
max: 107805333.0
negative: 0
zero: 163

--- buildingReplacementCost ---
min: 0.0
max: 155771643.0
negative: 0
zero: 163

--- buildingDamageAmount ---
min: 0.0
max: 3085581.0
negative: 0
zero: 164

--- contentsDamageAmount ---
min: 0.0
max: 1490364.0
negative: 0
zero: 404

--- waterDepth ---
min: -99.0
max: 162.0
nega

In [29]:
# Building damage greater than property value
check1 = df_clean[
    df_clean["buildingDamageAmount"] >
    df_clean["buildingPropertyValue"]
]

print("Damage > Property Value:", len(check1))

Damage > Property Value: 77


In [30]:
# Building damage greater than replacement cost
check2 = df_clean[
    df_clean["buildingDamageAmount"] >
    df_clean["buildingReplacementCost"]
]

print("Damage > Replacement Cost:", len(check2))

Damage > Replacement Cost: 73


In [31]:
# Building payment greater than building damage
check3 = df_clean[
    df_clean["amountPaidOnBuildingClaim"] >
    df_clean["buildingDamageAmount"]
]

print("Payment > Building Damage:", len(check3))

Payment > Building Damage: 416


In [32]:
# Building payment greater than building insurance coverage
check4 = df_clean[
    df_clean["amountPaidOnBuildingClaim"] >
    df_clean["totalBuildingInsuranceCoverage"]
]

print("Payment > Building Coverage:", len(check4))

Payment > Building Coverage: 21


In [33]:
check3[
    [
        "claim_ref",
        "buildingDamageAmount",
        "amountPaidOnBuildingClaim",
        "totalBuildingInsuranceCoverage",
        "netBuildingPaymentAmount",
        "source_system"
    ]
].head(20)

,claim_ref,buildingDamageAmount,amountPaidOnBuildingClaim,totalBuildingInsuranceCoverage,netBuildingPaymentAmount,source_system
5,CLM-7658990,27337.0,32195.94,144000.0,32195.94,core
11,CLM-6830121,6860.0,7841.52,250000.0,7841.52,core
13,CLM-7822961,59343.0,69572.93,250000.0,69572.93,core
24,CLM-6278537,63571.0,70563.40,250000.0,70563.40,core
27,CLM-6166678,12021.0,12592.37,250000.0,12592.37,core
43,CLM-7100608,122175.0,129784.55,250000.0,129784.55,core
45,CLM-7411284,15087.0,17023.65,250000.0,17023.65,core
50,CLM-6786774,22779.0,23893.23,250000.0,23893.23,core
60,CLM-7283680,155154.0,170596.62,250000.0,170596.62,core
88,CLM-5486841,28535.0,44000.00,22000.0,44000.00,core


## Issue
Found water depth and few claims as negative. 

Planned to replace negative claims with nan as it is not possible to have negative claims.

Retained water depth because I didn't have sufficient schema/domain evidence to prove they were invalid.

Set only the invalid latitude values to NaN as they were below -90.

In [34]:
df_clean.loc[df_clean["waterDepth"] < 0, "waterDepth"].value_counts()

waterDepth
-7.0     22
-1.0     16
-5.0     13
-8.0     10
-2.0     10
-6.0      9
-3.0      8
-9.0      8
-4.0      5
-99.0     1
-12.0     1
-10.0     1
-42.0     1
-89.0     1
-93.0     1
-57.0     1
-67.0     1
-20.0     1
Name: count, dtype: int64

In [35]:
df_clean[
    (df_clean["amountPaidOnBuildingClaim"] < 0) |
    (df_clean["amountPaidOnContentsClaim"] < 0) |
    (df_clean["netBuildingPaymentAmount"] < 0)
][[
    "claim_ref",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount",
    "nonPaymentReasonBuilding",
    "source_system"
]]

,claim_ref,amountPaidOnBuildingClaim,amountPaidOnContentsClaim,netBuildingPaymentAmount,nonPaymentReasonBuilding,source_system
330,CLM-7980100,-5.0,-1.0,-5.0,98.0,legacy_bdx


In [36]:
df_clean[
    (df_clean["latitude"] < -90) |
    (df_clean["latitude"] > 90)
][[
    "claim_ref", "latitude", "longitude", "state", "reportedZipCode"
]]

,claim_ref,latitude,longitude,state,reportedZipCode
582,CLM-8100313,-93.5,41.6,IA,50317
1064,CLM-7961163,-91.0,30.3,LA,70769
1855,CLM-6284078,-90.5,30.1,LA,70068


In [37]:
print("NEGATIVE WATER DEPTH")
print(df_clean.loc[df_clean["waterDepth"] < 0, "waterDepth"].value_counts())

print("\nNEGATIVE PAYMENTS")
print(df_clean[
    (df_clean["amountPaidOnBuildingClaim"] < 0) |
    (df_clean["amountPaidOnContentsClaim"] < 0) |
    (df_clean["netBuildingPaymentAmount"] < 0)
][[
    "claim_ref",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount",
    "nonPaymentReasonBuilding",
    "source_system"
]])

print("\nINVALID LATITUDE")
print(df_clean[
    (df_clean["latitude"] < -90) |
    (df_clean["latitude"] > 90)
][[
    "claim_ref", "latitude", "longitude",
    "state", "reportedZipCode", "source_system"
]])

print("\nPAYMENT > COVERAGE")
print(df_clean[
    df_clean["amountPaidOnBuildingClaim"] >
    df_clean["totalBuildingInsuranceCoverage"]
][[
    "claim_ref",
    "amountPaidOnBuildingClaim",
    "totalBuildingInsuranceCoverage",
    "buildingDamageAmount",
    "netBuildingPaymentAmount",
    "source_system"
]])

NEGATIVE WATER DEPTH
waterDepth
-7.0     22
-1.0     16
-5.0     13
-8.0     10
-2.0     10
-6.0      9
-3.0      8
-9.0      8
-4.0      5
-99.0     1
-12.0     1
-10.0     1
-42.0     1
-89.0     1
-93.0     1
-57.0     1
-67.0     1
-20.0     1
Name: count, dtype: int64

NEGATIVE PAYMENTS
       claim_ref  amountPaidOnBuildingClaim  amountPaidOnContentsClaim  \
330  CLM-7980100                       -5.0                       -1.0   

     netBuildingPaymentAmount  nonPaymentReasonBuilding source_system  
330                      -5.0                      98.0    legacy_bdx  

INVALID LATITUDE
        claim_ref  latitude  longitude state reportedZipCode source_system
582   CLM-8100313     -93.5       41.6    IA           50317          core
1064  CLM-7961163     -91.0       30.3    LA           70769          core
1855  CLM-6284078     -90.5       30.1    LA           70068          core

PAYMENT > COVERAGE
        claim_ref  amountPaidOnBuildingClaim  totalBuildingInsuranceCoverage

In [38]:
df_clean[
    df_clean["amountPaidOnBuildingClaim"].notna() &
    df_clean["netBuildingPaymentAmount"].notna()
].assign(
    difference=lambda x:
        x["amountPaidOnBuildingClaim"] -
        x["netBuildingPaymentAmount"]
)[[
    "claim_ref",
    "amountPaidOnBuildingClaim",
    "netBuildingPaymentAmount",
    "difference",
    "source_system"
]].sort_values(
    "difference",
    ascending=False
).head(20)


,claim_ref,amountPaidOnBuildingClaim,netBuildingPaymentAmount,difference,source_system
177,CLM-5544627,34912215.45,44586.42,34867629.03,core
1290,CLM-6925211,950091.17,2423.80,947667.37,core
138,CLM-5685223,601320.57,94471.19,506849.38,core
1554,CLM-8078474,501394.56,7124.44,494270.12,core
547,CLM-6292038,490633.32,5821.32,484812.00,core
727,CLM-6887750,481263.76,0.00,481263.76,core
1240,CLM-7798283,417022.49,0.00,417022.49,core
1976,CLM-7453133,416710.99,0.00,416710.99,core
482,CLM-5471799,471108.86,55251.15,415857.71,core
1110,CLM-5988109,494190.20,85213.31,408976.89,core


In [39]:
negative_water = df_clean[df_clean["waterDepth"] < 0]

print(
    negative_water[
        [
            "claim_ref",
            "waterDepth",
            "floodWaterDuration",
            "floodEvent",
            "causeOfDamage",
            "source_system"
        ]
    ].to_string(index=False)
)

    claim_ref  waterDepth  floodWaterDuration                       floodEvent causeOfDamage source_system
  CLM-7898726        -7.0                 NaN Late-September New York Flooding             4    legacy_bdx
  CLM-6776458        -8.0                 NaN                              NaN             4          core
  CLM-8074075        -7.0                 NaN                              NaN             4          core
  CLM-7218663        -3.0                 NaN                              NaN             4          core
  CLM-6494245        -2.0                 NaN                    Hurricane Ida             2          core
  CLM-5800002        -5.0                 NaN                 Hurricane Milton             1          core
  CLM-7240110        -7.0                 NaN                    Hurricane Ida             4          core
  CLM-8118865        -3.0                 NaN                    Hurricane Ida             4          core
  CLM-5492313        -7.0            

In [40]:
df_clean[df_clean["waterDepth"] < 0]["source_system"].value_counts()

source_system
core          104
legacy_bdx      6
Name: count, dtype: int64

In [41]:
for col in [
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount"
]:
    print(f"\n--- {col} ---")
    print(df_clean.loc[df_clean[col] < 0, col].value_counts())


--- amountPaidOnBuildingClaim ---
amountPaidOnBuildingClaim
-5.0    1
Name: count, dtype: int64

--- amountPaidOnContentsClaim ---
amountPaidOnContentsClaim
-1.0    1
Name: count, dtype: int64

--- netBuildingPaymentAmount ---
netBuildingPaymentAmount
-5.0    1
Name: count, dtype: int64


In [42]:
df_clean["nonPaymentReasonBuilding"].value_counts().sort_index()

nonPaymentReasonBuilding
1.0      70
6.0     110
7.0       2
8.0      10
15.0      1
16.0      2
20.0     38
97.0     62
98.0     20
99.0     74
Name: count, dtype: int64

# Clean 3
Fixing out of range latitude values


In [100]:
df_clean.loc[
    (df_clean["latitude"] < -90) |
    (df_clean["latitude"] > 90),
    "latitude"
] = float("nan")

In [44]:
negative_water = df_clean[df_clean["waterDepth"] < 0]

print(
    negative_water[
        [
            "claim_ref",
            "waterDepth",
            "floodWaterDuration",
            "floodEvent",
            "causeOfDamage",
            "source_system"
        ]
    ].to_string(index=False)
)

print("\nSource distribution:")
print(negative_water["source_system"].value_counts())

    claim_ref  waterDepth  floodWaterDuration                       floodEvent causeOfDamage source_system
  CLM-7898726        -7.0                 NaN Late-September New York Flooding             4    legacy_bdx
  CLM-6776458        -8.0                 NaN                              NaN             4          core
  CLM-8074075        -7.0                 NaN                              NaN             4          core
  CLM-7218663        -3.0                 NaN                              NaN             4          core
  CLM-6494245        -2.0                 NaN                    Hurricane Ida             2          core
  CLM-5800002        -5.0                 NaN                 Hurricane Milton             1          core
  CLM-7240110        -7.0                 NaN                    Hurricane Ida             4          core
  CLM-8118865        -3.0                 NaN                    Hurricane Ida             4          core
  CLM-5492313        -7.0            

In [45]:
payment_cols = [
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount",
    "buildingDamageAmount",
    "contentsDamageAmount",
    "totalBuildingInsuranceCoverage",
    "totalContentsInsuranceCoverage"
]

df_clean[payment_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
amountPaidOnBuildingClaim,1848.0,78058.030813,821422.920655,-5.0,3706.42500,19630.58,74619.7525,34912215.45
amountPaidOnContentsClaim,1846.0,7862.331450,27670.506900,-1.0,0.00000,0.00,2560.3175,500000.00
netBuildingPaymentAmount,2040.0,50727.792065,121328.409769,-5.0,129.08125,14011.53,64447.3450,3447017.17
buildingDamageAmount,1820.0,58027.127110,127779.985377,0.0,4895.50000,19553.00,74653.2500,3085581.00
contentsDamageAmount,1136.0,16286.448781,62229.299250,0.0,0.00000,1548.50,15043.7500,1490364.00
totalBuildingInsuranceCoverage,2040.0,306770.013824,986798.148457,14.0,148875.00000,250000.00,250000.0000,20165000.00
totalContentsInsuranceCoverage,2040.0,47986.998039,61190.785520,0.0,0.00000,30000.00,100000.0000,500000.00


In [46]:
check = df_clean[
    df_clean["netBuildingPaymentAmount"] >
    df_clean["amountPaidOnBuildingClaim"]
]

print("Net > Building payment:", len(check))

print(check[
    [
        "claim_ref",
        "amountPaidOnBuildingClaim",
        "amountPaidOnContentsClaim",
        "netBuildingPaymentAmount",
        "buildingDamageAmount",
        "source_system"
    ]
].head(30))

Net > Building payment: 0
Empty DataFrame
Columns: [claim_ref, amountPaidOnBuildingClaim, amountPaidOnContentsClaim, netBuildingPaymentAmount, buildingDamageAmount, source_system]
Index: []


In [47]:
print(
    "Net == building payment:",
    (
        df_clean["netBuildingPaymentAmount"] ==
        df_clean["amountPaidOnBuildingClaim"]
    ).sum()
)

print(
    "Net > 0 while building payment = 0:",
    (
        (df_clean["netBuildingPaymentAmount"] > 0) &
        (df_clean["amountPaidOnBuildingClaim"] == 0)
    ).sum()
)

Net == building payment: 1833
Net > 0 while building payment = 0: 0


In [48]:
damage_property = df_clean[
    df_clean["buildingDamageAmount"] >
    df_clean["buildingPropertyValue"]
].copy()

damage_property["damage_to_property_ratio"] = (
    damage_property["buildingDamageAmount"] /
    damage_property["buildingPropertyValue"].replace(0, float("nan"))
)

print(
    damage_property[
        [
            "claim_ref",
            "buildingDamageAmount",
            "buildingPropertyValue",
            "buildingReplacementCost",
            "totalBuildingInsuranceCoverage",
            "damage_to_property_ratio",
            "source_system"
        ]
    ].sort_values(
        "damage_to_property_ratio",
        ascending=False
    ).head(20)
)

        claim_ref  buildingDamageAmount  buildingPropertyValue  \
943   CLM-7185471           1088937.000             213573.000   
1221  CLM-6929169                78.213                 67.642   
447   CLM-6788159            163666.000             148000.000   
10    CLM-6726350            199577.000             180686.000   
1147  CLM-8150954             23772.000              21772.000   
1591  CLM-5459928            354174.000             348244.000   
12    CLM-7820778            191858.000                  0.000   
38    CLM-6601785             37970.000                  0.000   
76    CLM-6732637            107206.000                  0.000   
80    CLM-5826796             12165.000                  0.000   
118   CLM-8009009              6436.000                  0.000   
198   CLM-6967195            113858.000                  0.000   
207   CLM-6018275                 1.000                  0.000   
257   CLM-6488458             12000.000                  0.000   
280   CLM-

In [49]:
damage_replacement = df_clean[
    df_clean["buildingDamageAmount"] >
    df_clean["buildingReplacementCost"]
].copy()

damage_replacement["damage_to_replacement_ratio"] = (
    damage_replacement["buildingDamageAmount"] /
    damage_replacement["buildingReplacementCost"].replace(0, float("nan"))
)

print(
    damage_replacement[
        [
            "claim_ref",
            "buildingDamageAmount",
            "buildingReplacementCost",
            "buildingPropertyValue",
            "totalBuildingInsuranceCoverage",
            "damage_to_replacement_ratio",
            "source_system"
        ]
    ].sort_values(
        "damage_to_replacement_ratio",
        ascending=False
    ).head(20)
)

       claim_ref  buildingDamageAmount  buildingReplacementCost  \
943  CLM-7185471             1088937.0                 255329.0   
12   CLM-7820778              191858.0                      0.0   
38   CLM-6601785               37970.0                      0.0   
76   CLM-6732637              107206.0                      0.0   
80   CLM-5826796               12165.0                      0.0   
118  CLM-8009009                6436.0                      0.0   
134  CLM-6830799                 103.0                      0.0   
198  CLM-6967195              113858.0                      0.0   
207  CLM-6018275                   1.0                      0.0   
257  CLM-6488458               12000.0                      0.0   
280  CLM-6906720               48119.0                      0.0   
303  CLM-7330184                   1.0                      0.0   
311  CLM-5868777                   1.0                      0.0   
356  CLM-6519081                5000.0                      0.

In [50]:
zero_values = df_clean[
    (df_clean["buildingPropertyValue"] == 0) |
    (df_clean["buildingReplacementCost"] == 0)
].copy()

print("Rows:", len(zero_values))

print("\nSource system:")
print(zero_values["source_system"].value_counts())

print("\nBoth zero:")
print(
    (
        (zero_values["buildingPropertyValue"] == 0) &
        (zero_values["buildingReplacementCost"] == 0)
    ).sum()
)

print("\nOnly property value zero:")
print(
    (
        (zero_values["buildingPropertyValue"] == 0) &
        (zero_values["buildingReplacementCost"] != 0)
    ).sum()
)

print("\nOnly replacement cost zero:")
print(
    (
        (zero_values["buildingPropertyValue"] != 0) &
        (zero_values["buildingReplacementCost"] == 0)
    ).sum()
)

Rows: 167

Source system:
source_system
core          158
legacy_bdx      9
Name: count, dtype: int64

Both zero:
159

Only property value zero:
4

Only replacement cost zero:
4


In [51]:
df_clean[
    df_clean["claim_ref"] == "CLM-7185471"
].T

,943
claim_ref,CLM-7185471
source_system,core
dateOfLoss,2022-07-26 00:00:00+00:00
yearOfLoss,2022
state,MO
reportedZipCode,63043
countyCode,29189.0
latitude,38.7
longitude,-90.4
floodEvent,Late July Flooding


In [52]:
print("Building damage < $100:")
print(
    df_clean[
        df_clean["buildingDamageAmount"].notna() &
        (df_clean["buildingDamageAmount"] < 100)
    ][[
        "claim_ref",
        "buildingDamageAmount",
        "buildingPropertyValue",
        "buildingReplacementCost",
        "amountPaidOnBuildingClaim",
        "source_system"
    ]].to_string(index=False)
)

Building damage < $100:
    claim_ref  buildingDamageAmount  buildingPropertyValue  buildingReplacementCost  amountPaidOnBuildingClaim source_system
  CLM-5642877                 0.000                  0.000                    0.000                      0.000          core
  CLM-5976757                87.000              86092.000               131458.000                      0.000          core
  CLM-5576313                 0.000                  0.000                    0.000                        NaN          core
  CLM-7455343                 0.000             132438.000               228342.000                      0.000          core
  CLM-6223381                 0.000             293000.000               329700.000                      0.000          core
  CLM-5637999                 0.000             150000.000               150000.000                        NaN          core
  CLM-7806166                 0.000             311295.000               371882.000                  

In [53]:
print("Property value < $100:")
print(
    df_clean[
        df_clean["buildingPropertyValue"].notna() &
        (df_clean["buildingPropertyValue"] < 100)
    ][[
        "claim_ref",
        "buildingPropertyValue",
        "buildingReplacementCost",
        "buildingDamageAmount",
        "source_system"
    ]].to_string(index=False)
)

Property value < $100:
    claim_ref  buildingPropertyValue  buildingReplacementCost  buildingDamageAmount source_system
  CLM-5642877                  0.000                    0.000                 0.000          core
  CLM-7820778                  0.000                    0.000            191858.000          core
  CLM-5576313                  0.000                    0.000                 0.000          core
CLM-6839569-R                  0.000                    0.000                 0.000          core
  CLM-6601785                  0.000                    0.000             37970.000          core
  CLM-7736420                  0.000                    0.000                 0.000          core
  CLM-6732637                  0.000                    0.000            107206.000          core
  CLM-5826796                  0.000                    0.000             12165.000          core
  CLM-7948788                  0.000                    0.000                 0.000          co

In [54]:
money_cols = [
    "totalBuildingInsuranceCoverage",
    "totalContentsInsuranceCoverage",
    "buildingPropertyValue",
    "buildingReplacementCost",
    "buildingDamageAmount",
    "contentsDamageAmount",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount"
]

df_clean.groupby("source_system")[money_cols].median().T

source_system,core,legacy_bdx
totalBuildingInsuranceCoverage,250000.00,250.000
totalContentsInsuranceCoverage,40000.00,46.000
buildingPropertyValue,191799.00,181.113
buildingReplacementCost,251887.00,234.752
buildingDamageAmount,23372.00,23.697
contentsDamageAmount,2318.00,6.314
amountPaidOnBuildingClaim,23405.74,21.933
amountPaidOnContentsClaim,0.00,0.000
netBuildingPaymentAmount,16658.82,16.385


## Clean 4
Monetary values in legacy_bdx are stored in thousands, while values in core are stored in dollars.

In [55]:
monetary_cols = [
    "totalBuildingInsuranceCoverage",
    "totalContentsInsuranceCoverage",
    "buildingPropertyValue",
    "buildingReplacementCost",
    "buildingDamageAmount",
    "contentsDamageAmount",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount"
]

legacy_mask = df_clean["source_system"] == "legacy_bdx"

df_clean.loc[legacy_mask, monetary_cols] = (
    df_clean.loc[legacy_mask, monetary_cols] * 1000
)

In [56]:
df_clean.groupby("source_system")[monetary_cols].median().T

source_system,core,legacy_bdx
totalBuildingInsuranceCoverage,250000.00,250000.0
totalContentsInsuranceCoverage,40000.00,46000.0
buildingPropertyValue,191799.00,181113.0
buildingReplacementCost,251887.00,234752.0
buildingDamageAmount,23372.00,23697.0
contentsDamageAmount,2318.00,6314.0
amountPaidOnBuildingClaim,23405.74,21933.0
amountPaidOnContentsClaim,0.00,0.0
netBuildingPaymentAmount,16658.82,16385.0


In [57]:
print("Legacy rows:", legacy_mask.sum())

print(
    df_clean.groupby("source_system")[
        ["totalBuildingInsuranceCoverage", "buildingDamageAmount"]
    ].median()
)

Legacy rows: 123
               totalBuildingInsuranceCoverage  buildingDamageAmount
source_system                                                      
core                                 250000.0               23372.0
legacy_bdx                           250000.0               23697.0


In [58]:
checks = {
    "damage_gt_property": (
        df_clean["buildingDamageAmount"] >
        df_clean["buildingPropertyValue"]
    ),
    "damage_gt_replacement": (
        df_clean["buildingDamageAmount"] >
        df_clean["buildingReplacementCost"]
    ),
    "payment_gt_coverage": (
        df_clean["amountPaidOnBuildingClaim"] >
        df_clean["totalBuildingInsuranceCoverage"]
    ),
    "negative_building_payment": (
        df_clean["amountPaidOnBuildingClaim"] < 0
    )
}

for name, mask in checks.items():
    print(name, mask.sum())

damage_gt_property 77
damage_gt_replacement 73
payment_gt_coverage 21
negative_building_payment 1


In [59]:
damage_value_issues = df_clean[
    (df_clean["buildingDamageAmount"] > df_clean["buildingPropertyValue"]) |
    (df_clean["buildingDamageAmount"] > df_clean["buildingReplacementCost"])
][[
    "claim_ref",
    "source_system",
    "state",
    "dateOfLoss",
    "buildingPropertyValue",
    "buildingReplacementCost",
    "buildingDamageAmount",
    "amountPaidOnBuildingClaim",
    "totalBuildingInsuranceCoverage"
]].copy()

damage_value_issues

,claim_ref,source_system,state,dateOfLoss,buildingPropertyValue,buildingReplacementCost,buildingDamageAmount,amountPaidOnBuildingClaim,totalBuildingInsuranceCoverage
10,CLM-6726350,core,FL,2024-09-26 00:00:00+00:00,180686.0,358376.0,199577.0,189576.83,250000.0
12,CLM-7820778,core,FL,2024-09-27 00:00:00+00:00,0.0,0.0,191858.0,189857.65,250000.0
38,CLM-6601785,core,LA,2021-08-29 00:00:00+00:00,0.0,0.0,37970.0,36719.75,223000.0
76,CLM-6732637,core,FL,2022-09-28 00:00:00+00:00,0.0,0.0,107206.0,102205.57,250000.0
80,CLM-5826796,core,FL,2024-09-26 00:00:00+00:00,0.0,0.0,12165.0,10164.64,250000.0
118,CLM-8009009,core,KY,2021-03-08 00:00:00+00:00,0.0,0.0,6436.0,0.00,200000.0
134,CLM-6830799,core,TX,2019-09-17 00:00:00+00:00,149284.0,0.0,103.0,0.00,250000.0
198,CLM-6967195,core,FL,2024-09-26 00:00:00+00:00,0.0,0.0,113858.0,121230.38,149000.0
207,CLM-6018275,core,TX,2023-10-28 00:00:00+00:00,0.0,0.0,1.0,0.00,250000.0
257,CLM-6488458,core,FL,2024-09-27 00:00:00+00:00,0.0,0.0,12000.0,240936.03,250000.0


## Clean 5
Replaced 0 buildingPropertyValue with nan. 

Replaced 0 buildingReplacementCost with nan.

In [60]:
import numpy as np

In [61]:
df_clean.loc[
    df_clean["buildingPropertyValue"] == 0,
    "buildingPropertyValue"
] = np.nan

df_clean.loc[
    df_clean["buildingReplacementCost"] == 0,
    "buildingReplacementCost"
] = np.nan

### Several business-logic inconsistencies were identified, including damage exceeding reported property/replacement values and payments exceeding reported coverage. I retained these because the available fields do not establish that these relationships are strict constraints. Modifying them would risk fabricating values.

In [62]:
df_clean[
    (df_clean["amountPaidOnBuildingClaim"] < 0) |
    (df_clean["amountPaidOnContentsClaim"] < 0) |
    (df_clean["netBuildingPaymentAmount"] < 0)
][[
    "claim_ref",
    "source_system",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount",
    "nonPaymentReasonBuilding"
]]

,claim_ref,source_system,amountPaidOnBuildingClaim,amountPaidOnContentsClaim,netBuildingPaymentAmount,nonPaymentReasonBuilding
330,CLM-7980100,legacy_bdx,-5000.0,-1000.0,-5000.0,98.0


In [72]:
base_ref = df_clean["claim_ref"].str.replace("-R$", "", regex=True)

df_clean[base_ref.duplicated(keep=False)][[
    "claim_ref",
    "source_system",
    "dateOfLoss",
    "state",
    "buildingDamageAmount",
    "amountPaidOnBuildingClaim"
]].sort_values("claim_ref")

,claim_ref,source_system,dateOfLoss,state,buildingDamageAmount,amountPaidOnBuildingClaim
1323,CLM-5457646,core,2021-08-29 00:00:00+00:00,NY,7269.0,6019.22
1711,CLM-5457646-R,core,2021-08-29 00:00:00+00:00,NY,7269.0,6019.22
1167,CLM-5589318,core,2023-08-30 00:00:00+00:00,FL,87790.0,92013.12
1775,CLM-5589318-R,core,2023-08-30 00:00:00+00:00,FL,87790.0,92013.12
1676,CLM-5598359,legacy_bdx,NaT,CA,0.0,0.00
1211,CLM-5598359-R,legacy_bdx,NaT,CA,0.0,0.00
763,CLM-5615193,core,2018-09-14 00:00:00+00:00,NC,3034.0,1784.72
1694,CLM-5615193-R,core,2018-09-14 00:00:00+00:00,NC,3034.0,1784.72
1005,CLM-5690103,core,2021-01-15 00:00:00+00:00,MD,NaN,NaN
1074,CLM-5690103-R,core,2021-01-15 00:00:00+00:00,MD,NaN,NaN


## Clean 6
Replacing negative amoutnPaid col with nan


In [66]:
neg_payment_mask = (
    (df_clean["amountPaidOnBuildingClaim"] < 0) |
    (df_clean["amountPaidOnContentsClaim"] < 0) |
    (df_clean["netBuildingPaymentAmount"] < 0)
)

df_clean.loc[neg_payment_mask, [
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "netBuildingPaymentAmount"
]] = np.nan

In [69]:
df_clean["base_claim_ref"] = df_clean["claim_ref"].str.replace(
    r"-R$",
    "",
    regex=True
)

base_counts = df_clean["base_claim_ref"].value_counts()

base_counts[base_counts > 1].head(200)

base_claim_ref
CLM-6839569    2
CLM-6678785    2
CLM-7433935    2
CLM-8057769    2
CLM-7127665    2
CLM-7495589    2
CLM-8140690    2
CLM-6521778    2
CLM-6053137    2
CLM-7124335    2
CLM-8106199    2
CLM-6229538    2
CLM-8109648    2
CLM-7773144    2
CLM-6597136    2
CLM-6658432    2
CLM-8041087    2
CLM-5615193    2
CLM-7804614    2
CLM-7449518    2
CLM-6145841    2
CLM-8004037    2
CLM-7468945    2
CLM-6576331    2
CLM-5690103    2
CLM-6881275    2
CLM-7446191    2
CLM-6613036    2
CLM-5589318    2
CLM-6581402    2
CLM-5598359    2
CLM-7621754    2
CLM-5457646    2
CLM-8072327    2
CLM-6268837    2
CLM-6009600    2
CLM-6888019    2
CLM-7936455    2
CLM-6002479    2
CLM-7685251    2
Name: count, dtype: int64

## Clean 7
Removing dups

In [73]:
r_mask = df_clean["claim_ref"].str.endswith("-R")

print("R duplicate rows:", r_mask.sum())

df_clean = df_clean[~r_mask].copy()

print("Rows after removing duplicates:", len(df_clean))

R duplicate rows: 40
Rows after removing duplicates: 2000


In [74]:
df_clean["claim_ref"].str.endswith("-R").sum()

np.int64(0)

In [75]:
for col in [
    "causeOfDamage",
    "ratedFloodZone",
    "occupancyType",
    "nonPaymentReasonBuilding"
]:
    print("\n", col)
    print(df_clean[col].value_counts(dropna=False))


 causeOfDamage
causeOfDamage
4      892
1      446
2      394
0      237
NaN     27
D        3
3        1
Name: count, dtype: int64

 ratedFloodZone
ratedFloodZone
AE     1270
X       437
A        93
VE       65
AH       31
C        25
AO       21
B        11
NaN       5
A08       5
A99       5
A13       4
A10       4
A09       3
A04       3
A05       2
A15       2
A07       2
A01       2
A12       2
AHB       1
A11       1
A18       1
A02       1
V16       1
AOB       1
D         1
A03       1
Name: count, dtype: int64

 occupancyType
occupancyType
11    1279
1      306
18     118
15      92
12      61
13      35
14      30
16      27
6       25
3       14
2       10
4        3
Name: count, dtype: int64

 nonPaymentReasonBuilding
nonPaymentReasonBuilding
NaN     1616
6.0      108
99.0      74
1.0       70
97.0      60
20.0      38
98.0      19
8.0       10
16.0       2
7.0        2
15.0       1
Name: count, dtype: int64


In [76]:
missing = (
    df_clean.isna()
    .sum()
    .to_frame("missing")
)

missing["missing_pct"] = missing["missing"] / len(df_clean) * 100

missing.sort_values("missing_pct", ascending=False)

,missing,missing_pct
nonPaymentReasonBuilding,1616,80.80
floodWaterDuration,1579,78.95
contentsDamageAmount,886,44.30
floodEvent,551,27.55
buildingPropertyValue,375,18.75
buildingReplacementCost,375,18.75
waterDepth,305,15.25
buildingDamageAmount,218,10.90
mostRecentPaymentDate,197,9.85
amountPaidOnContentsClaim,192,9.60


In [77]:
df_clean[df_clean["dateOfLoss"].isna()]["source_system"].value_counts()

source_system
legacy_bdx    120
Name: count, dtype: int64

In [83]:
df_clean.loc[
    df_clean["dateOfLoss"].isna(),
    ["claim_ref", "source_system", "dateOfLoss"]
]

,claim_ref,source_system,dateOfLoss
40,CLM-7898726,legacy_bdx,NaT
67,CLM-5746454,legacy_bdx,NaT
82,CLM-6157209,legacy_bdx,NaT
99,CLM-6470168,legacy_bdx,NaT
144,CLM-6398916,legacy_bdx,NaT
154,CLM-7011673,legacy_bdx,NaT
156,CLM-5725175,legacy_bdx,NaT
158,CLM-5848380,legacy_bdx,NaT
165,CLM-8169782,legacy_bdx,NaT
168,CLM-6916300,legacy_bdx,NaT


In [82]:
df.loc[
    df["claim_ref"].isin(missing_loss_claims),
    ["claim_ref", "source_system", "dateOfLoss"]
]

,claim_ref,source_system,dateOfLoss
40,CLM-7898726,legacy_bdx,29-09-2023
67,CLM-5746454,legacy_bdx,30-08-2023
82,CLM-6157209,legacy_bdx,10-10-2024
99,CLM-6470168,legacy_bdx,09-10-2024
144,CLM-6398916,legacy_bdx,30-09-2022
154,CLM-7011673,legacy_bdx,05-04-2025
156,CLM-5725175,legacy_bdx,23-03-2026
158,CLM-5848380,legacy_bdx,30-09-2022
165,CLM-8169782,legacy_bdx,19-05-2020
168,CLM-6916300,legacy_bdx,21-04-2025


In [86]:
raw_loss = df["dateOfLoss"].astype("string").str.strip()

parsed_loss = pd.Series(
    pd.NaT,
    index=df.index,
    dtype="datetime64[ns, UTC]"
)

# Core: YYYY-MM-DDTHH:MM:SS.sssZ
core_mask = df["source_system"].eq("core")

parsed_loss.loc[core_mask] = pd.to_datetime(
    raw_loss.loc[core_mask],
    format="%Y-%m-%dT%H:%M:%S.%fZ",
    errors="coerce",
    utc=True
)

# Legacy: DD-MM-YYYY
legacy_mask = df["source_system"].eq("legacy_bdx")

parsed_loss.loc[legacy_mask] = pd.to_datetime(
    raw_loss.loc[legacy_mask],
    format="%d-%m-%Y",
    errors="coerce",
    utc=True
)

# Map back to the cleaned dataframe
date_map = pd.Series(
    parsed_loss.values,
    index=df["claim_ref"]
)

df_clean["dateOfLoss"] = df_clean["claim_ref"].map(date_map)

In [87]:
print("Missing dateOfLoss:", df_clean["dateOfLoss"].isna().sum())
print(df_clean["dateOfLoss"].min())
print(df_clean["dateOfLoss"].max())

Missing dateOfLoss: 0
2018-01-04 00:00:00
2026-08-20 00:00:00


In [88]:
print("Shape:", df_clean.shape)

print("\nDuplicate rows:", df_clean.duplicated().sum())
print("Duplicate claim_ref:", df_clean["claim_ref"].duplicated().sum())

print("\nMissing dateOfLoss:", df_clean["dateOfLoss"].isna().sum())
print("Missing openDate:", df_clean["openDate"].isna().sum())

print("\nInvalid latitude:",
      ((df_clean["latitude"] < -90) |
       (df_clean["latitude"] > 90)).sum())

print("\nNegative payments:")
print(
    (
        (df_clean["amountPaidOnBuildingClaim"] < 0) |
        (df_clean["amountPaidOnContentsClaim"] < 0) |
        (df_clean["netBuildingPaymentAmount"] < 0)
    ).sum()
)

Shape: (2000, 32)

Duplicate rows: 0
Duplicate claim_ref: 0

Missing dateOfLoss: 0
Missing openDate: 0

Invalid latitude: 0

Negative payments:
0


In [91]:
date_cols = [
    "dateOfLoss",
    "openDate",
    "originalConstructionDate",
    "mostRecentPaymentDate"
]

for col in date_cols:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce",
        utc=True
    )

In [92]:
print(
    "Payment before loss:",
    (df_clean["mostRecentPaymentDate"] < df_clean["dateOfLoss"]).sum()
)

print(
    "Construction after loss:",
    (df_clean["originalConstructionDate"] > df_clean["dateOfLoss"]).sum()
)

print(
    "Open date before loss:",
    (df_clean["openDate"] < df_clean["dateOfLoss"]).sum()
)

Payment before loss: 0
Construction after loss: 0
Open date before loss: 0


In [93]:
df_clean[
    df_clean["ratedFloodZone"].isin(["AHB", "AOB", "D"])
][[
    "claim_ref",
    "state",
    "reportedZipCode",
    "ratedFloodZone",
    "floodEvent"
]]

,claim_ref,state,reportedZipCode,ratedFloodZone,floodEvent
66,CLM-7986783,TX,78503,AHB,NaN
879,CLM-7990756,CA,93065,AOB,NaN
1397,CLM-7651852,CA,91040,D,California Atmospheric River


In [94]:
df_clean = df_clean.drop(columns=["base_claim_ref"])

In [98]:
print((df_clean["longitude"] < -90).any())

True


In [95]:
# Make sure temporary investigation columns are removed
if "base_claim_ref" in df_clean.columns:
    df_clean = df_clean.drop(columns=["base_claim_ref"])

# Save cleaned dataset
df_clean.to_csv("nfip_claims_cleaned.csv", index=False)

print("Saved cleaned dataset")
print("Shape:", df_clean.shape)

Saved cleaned dataset
Shape: (2000, 31)


In [97]:
print("Shape:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate claim_ref:", df_clean["claim_ref"].duplicated().sum())

print("Missing dateOfLoss:", df_clean["dateOfLoss"].isna().sum())
print("Missing openDate:", df_clean["openDate"].isna().sum())

print(
    "Invalid latitude:",
    ((df_clean["latitude"] < -90) | (df_clean["latitude"] > 90)).sum()
)

print(
    "Negative payments:",
    (
        (df_clean["amountPaidOnBuildingClaim"] < 0) |
        (df_clean["amountPaidOnContentsClaim"] < 0) |
        (df_clean["netBuildingPaymentAmount"] < 0)
    ).sum()
)

print(
    "Payment before loss:",
    (
        df_clean["mostRecentPaymentDate"] < df_clean["dateOfLoss"]
    ).sum()
)

print(
    "Construction after loss:",
    (
        df_clean["originalConstructionDate"] > df_clean["dateOfLoss"]
    ).sum()
)

Shape: (2000, 31)
Duplicate rows: 0
Duplicate claim_ref: 0
Missing dateOfLoss: 0
Missing openDate: 0
Invalid latitude: 0
Negative payments: 0
Payment before loss: 0
Construction after loss: 0
